In [ ]:
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=200_000,
    unit="tokens",
    expires_in="24h",
    intent_context="multi-model research session",
    allocations=[
        {"category": "gpt4o",   "limit": 80_000,  "enforcement_mode": "CATEGORY_CONSTRAINED", "allowed_categories": ["gpt4o"]},
        {"category": "claude",  "limit": 100_000, "enforcement_mode": "CATEGORY_CONSTRAINED", "allowed_categories": ["claude"]},
        {"category": "o1",      "limit": 20_000,  "enforcement_mode": "CATEGORY_CONSTRAINED", "allowed_categories": ["o1"]},
    ],
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total:   {budget.total_limit:,} tokens")
print(f"  GPT-4o:  80,000  Claude: 100,000  o1: 20,000")

In [ ]:
# GPT-4o call — within allocation
auth = client.authorize(
    session_token=budget.session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="GPT-4o: summarize 10 papers",
    requested_quantity=35_000,
    claimed_category="gpt4o",
    idempotency_key="gpt-001",
)
print(f"GPT-4o call:  {auth.decision} — {auth.approved_quantity:,} tokens")
client.confirm_event(auth.event_id, confirmed_quantity=34_200)
print("✓ Confirmed.")

# o1 call — exceeds o1 allocation (20k limit)
auth2 = client.authorize(
    session_token=budget.session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="o1: deep reasoning task",
    requested_quantity=25_000,
    claimed_category="o1",
    idempotency_key="o1-001",
)
print(f"o1 call:      {auth2.decision} — {auth2.denial_reason}")
print("o1 allocation exhausted. Claude allocation untouched.")

# Claude call — within its separate allocation
auth3 = client.authorize(
    session_token=budget.session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="Claude: full synthesis",
    requested_quantity=60_000,
    claimed_category="claude",
    idempotency_key="claude-001",
)
print(f"Claude call:  {auth3.decision} — {auth3.approved_quantity:,} tokens")

print(f"\n→ See the spend tree: https://figuard-sandbox-1.onrender.com/ui")